# 94 — Kaggle GPU: MolFormer-XL Fine-tuning

IBM MolFormer-XL is pretrained on 1.1 billion SMILES (ZINC + PubChem).
Architecture: Linear Attention Transformer, 12 layers, 768-dim embeddings.

Strategy:
1. Load `ibm/MolFormer-XL-both-10pct` from HuggingFace
2. Extract [CLS] embedding for all compounds
3. Fine-tune: add regression head (3-layer MLP, dropout=0.3)
4. Scaffold 5-fold CV on GPU

Run on Kaggle: `python scripts/kaggle_push.py --nb 94 --pull`

In [ ]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
for p in ["/kaggle/input/pxr-challenge-data/src", "../src"]:
    if os.path.exists(p): sys.path.insert(0, p); break
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

if os.path.exists("/kaggle"):
    DATA_PROC   = Path("/kaggle/input/pxr-challenge-data/data/processed")
    SUBMISSIONS = Path("/kaggle/working")
else:
    sys.path.insert(0, "../src")
    from pxr.paths import DATA_PROCESSED as DATA_PROC, SUBMISSIONS

SEED = 42; torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  CUDA: {torch.cuda.is_available()}")


In [ ]:
from pxr.data import load_train, load_test
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko

tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, 5, SEED)
print(f"Train {len(tr):,}  Test {len(te):,}")


In [ ]:
try:
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained("ibm/MolFormer-XL-both-10pct", trust_remote_code=True)
    base = AutoModel.from_pretrained("ibm/MolFormer-XL-both-10pct", trust_remote_code=True)
    print(f"MolFormer loaded: {sum(p.numel() for p in base.parameters()):,} params")
    MOLFORMER_AVAIL = True
except Exception as e:
    print(f"MolFormer not available: {e}")
    MOLFORMER_AVAIL = False


In [ ]:
if MOLFORMER_AVAIL:
    class MolFormerRegressor(nn.Module):
        def __init__(self, encoder, d_model=768, dropout=0.3):
            super().__init__()
            self.encoder = encoder
            self.head = nn.Sequential(
                nn.Linear(d_model, 256), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(256, 64),  nn.GELU(), nn.Dropout(dropout/2),
                nn.Linear(64, 1)
            )
        def forward(self, input_ids, attention_mask):
            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            cls = out.last_hidden_state[:, 0, :]
            return self.head(cls).squeeze(-1)

    def tokenize_batch(smiles_list, max_len=202):
        return tok(smiles_list, return_tensors="pt", padding=True,
                   truncation=True, max_length=max_len)

    from torch.utils.data import Dataset, DataLoader
    class SMILESDataset(Dataset):
        def __init__(self, smiles, labels=None):
            self.smiles = smiles
            self.labels = labels
        def __len__(self): return len(self.smiles)
        def __getitem__(self, i):
            return self.smiles[i], self.labels[i] if self.labels is not None else float("nan")

    EPOCHS = 20; BATCH = 32; LR = 2e-5; PATIENCE = 5

    oof_molformer = np.full(len(y_tr), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        print(f"\n=== Fold {fold+1}/5 ===", flush=True)
        model_mf = MolFormerRegressor(base).to(device)
        opt = torch.optim.AdamW(model_mf.parameters(), lr=LR, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
        loss_fn = nn.MSELoss()

        tr_smi = tr.iloc[tr_idx]["smiles"].tolist()
        va_smi = tr.iloc[va_idx]["smiles"].tolist()
        tr_y_f = torch.tensor(y_tr[tr_idx], dtype=torch.float32)
        va_y_f = torch.tensor(y_tr[va_idx], dtype=torch.float32)

        best_val = float("inf"); patience_cnt = 0
        for epoch in range(EPOCHS):
            model_mf.train()
            perm = torch.randperm(len(tr_smi))
            epoch_loss = 0
            for b in range(0, len(tr_smi), BATCH):
                idx_b = perm[b:b+BATCH].tolist()
                enc = tokenize_batch([tr_smi[i] for i in idx_b])
                enc = {k: v.to(device) for k, v in enc.items()}
                pred = model_mf(**enc)
                loss = loss_fn(pred, tr_y_f[idx_b].to(device))
                opt.zero_grad(); loss.backward(); opt.step()
                epoch_loss += loss.item()
            sched.step()

            model_mf.eval()
            with torch.no_grad():
                va_preds = []
                for b in range(0, len(va_smi), BATCH*2):
                    enc = tokenize_batch(va_smi[b:b+BATCH*2])
                    enc = {k: v.to(device) for k, v in enc.items()}
                    va_preds.append(model_mf(**enc).cpu().numpy())
                va_pred = np.concatenate(va_preds)
                val_rae = rae(y_tr[va_idx], va_pred)
                print(f"  Epoch {epoch+1:3d}  train_loss={epoch_loss/max(1,len(tr_smi)//BATCH):.4f}  val_RAE={val_rae:.4f}", flush=True)
                if val_rae < best_val:
                    best_val = val_rae; patience_cnt = 0
                    best_preds = va_pred.copy()
                else:
                    patience_cnt += 1
                    if patience_cnt >= PATIENCE: break

        oof_molformer[va_idx] = best_preds
        print(f"Fold {fold+1} best RAE: {best_val:.4f}")

    oof_rae = rae(y_tr, oof_molformer)
    print(f"\nMolFormer OOF RAE: {oof_rae:.4f}")
else:
    print("MolFormer not available — saving placeholder")
    oof_molformer = np.full(len(y_tr), np.nan)


In [ ]:
if MOLFORMER_AVAIL and np.isfinite(oof_molformer).all():
    # Final model on all data
    model_final = MolFormerRegressor(base).to(device)
    opt = torch.optim.AdamW(model_final.parameters(), lr=LR, weight_decay=1e-4)
    all_smi = tr["smiles"].tolist()
    for epoch in range(EPOCHS):
        model_final.train()
        perm = torch.randperm(len(all_smi))
        for b in range(0, len(all_smi), BATCH):
            idx_b = perm[b:b+BATCH].tolist()
            enc = tokenize_batch([all_smi[i] for i in idx_b])
            enc = {k: v.to(device) for k, v in enc.items()}
            pred = model_final(**enc)
            loss = nn.MSELoss()(pred, torch.tensor(y_tr[idx_b], dtype=torch.float32).to(device))
            opt.zero_grad(); loss.backward(); opt.step()

    model_final.eval()
    te_smi = te["smiles"].tolist()
    te_preds_raw = []
    with torch.no_grad():
        for b in range(0, len(te_smi), BATCH*2):
            enc = tokenize_batch(te_smi[b:b+BATCH*2])
            enc = {k: v.to(device) for k, v in enc.items()}
            te_preds_raw.append(model_final(**enc).cpu().numpy())
    te_preds = np.clip(np.concatenate(te_preds_raw)[:513], y_tr.min()-0.5, y_tr.max()+0.5)
else:
    te_preds = np.full(513, y_tr.mean())

np.save(DATA_PROC/"oof_molformer_finetune.npy", oof_molformer)
np.save(DATA_PROC/"te_oof_molformer_finetune.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"94_molformer_finetune.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
